# Automatic Lyrics Transcription for 1-3-part singing
# Vocal separation > Singing Source Separation > ASR

1) **model_bs_roformer_ep_317_sdr_12.9755.ckpt**
2) **sepformer-libri2mix** for 2 voices, **sepformer-libri3mix** for 3 voices (none for solo singing)
3) **Qwen3ASR-1.7B**

In [ ]:
# @title ### Pre-installations

!pip install "audio-separator[gpu]" speechbrain qwen_asr
!pip install --upgrade evaluate jiwer

#@markdown Restart runtime afterwards

In [ ]:
# @title ### Loading the models

from audio_separator.separator import Separator
import os
import logging

# Source Separation for separating vocals from the other sound
separator1 = Separator()

separator1.load_model(model_filename="model_bs_roformer_ep_317_sdr_12.9755.ckpt")

output_names = {
    "Vocals": "vocals_output",
    "Instrumental": "instrumental_output",
}

from speechbrain.inference.separation import SepformerSeparation as separator
import torchaudio

# Source separation for separating vocal parts one from each other
model_libri2mix = separator.from_hparams(source="speechbrain/sepformer-libri2mix", savedir='pretrained_models/sepformer-libri2mix', run_opts={"device":"cuda"})
model_libri3mix = separator.from_hparams(source="speechbrain/sepformer-libri3mix", savedir='pretrained_models/sepformer-libri3mix', run_opts={"device":"cuda"})

import torch
from qwen_asr import Qwen3ASRModel

# Automatic Speech Recognition
model_qwen3asr = Qwen3ASRModel.from_pretrained(
        "Qwen/Qwen3-ASR-1.7B",
        dtype=torch.bfloat16,
        device_map="cuda:0",
        max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
        max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
    )

from pydub import AudioSegment
from pydub.utils import make_chunks

import warnings
import logging

warnings.filterwarnings('ignore')

# Function to aggressively set logger level and its handlers
def set_logger_level_aggressively(logger_name, level):
    logger = logging.getLogger(logger_name)
    logger.setLevel(level)
    # Ensure all handlers also respect this level
    for handler in logger.handlers:
        handler.setLevel(level)

# Set levels for specific libraries to suppress verbose output
set_logger_level_aggressively('audio_separator', logging.CRITICAL) # Set to CRITICAL for maximum suppression
set_logger_level_aggressively('speechbrain', logging.CRITICAL)   # Set to CRITICAL
set_logger_level_aggressively('transformers', logging.CRITICAL) # Set to CRITICAL

# Set the root logger's level to CRITICAL to suppress all INFO, DEBUG, WARNING, ERROR messages
set_logger_level_aggressively(None, logging.CRITICAL) # None refers to the root logger

In [ ]:
from IPython.display import Audio, display
import os

# @title ## Please cut your audio if it's longer than 20 seconds!

# User inputs for audio file and language
audio_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

output_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

start_time = "02:54:00" # @param {type:"string"}
end_time = "03:10:00" # @param {type:"string"}

def parse_time_to_ms(time_str):
    """
    Parses a time string in 'MM:SS:HH' format to milliseconds.
    HH represents hundredths of a second.
    """
    parts = time_str.split(':')
    minutes = int(parts[0])
    seconds = int(parts[1])
    hundredths = int(parts[2])
    total_ms = (minutes * 60 * 1000) + (seconds * 1000) + (hundredths * 10)
    return total_ms

def trim_audio(audio_path, start_time_str, end_time_str):
    """
    Trims an audio file from a given start time to an end time.

    Args:
        audio_path (str): Path to the input audio file.
        start_time_str (str): Start time in 'MM:SS:HH' format (e.g., '00:04:00').
        end_time_str (str): End time in 'MM:SS:HH' format (e.g., '00:05:30').

    Returns:
        pydub.AudioSegment: The trimmed audio segment.
    """
    try:
        audio = AudioSegment.from_file(audio_path)

        start_ms = parse_time_to_ms(start_time_str)
        end_ms = parse_time_to_ms(end_time_str)

        trimmed_audio = audio[start_ms:end_ms]
        return trimmed_audio

    except Exception as e:
        print(f"Произошла ошибка при обрезке аудио: {e}")
        return None

trim_audio(audio_file, start_time if start_time else "00:00:00", end_time if end_time else "00:20:00").export(output_file if output_file else "/content/cut_audio.wav", format="wav")

if os.path.exists(output_file):
  display(Audio(output_file))
else:
  print(f"Error: Audio file not found at {output_file}")

In [ ]:
# @title # Transcribe your song with 1, 2 or 3 voices!

import os
import shutil # Required for creating and clearing directories

# User inputs for audio file and language
audio_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

#@markdown ---
language = ""  # @param {type:"string"}
num_voices = 3 # @param ["1", "2", "3"] {type:"raw"}

# --- Output Management Logic ---
# Derive base name from audio_file (e.g., 'Pur_ti_miro_cut')
audio_base_name = os.path.splitext(os.path.basename(audio_file))[0]

# Define the output directory name based on audio and number of voices
output_folder_name = f"{audio_base_name}_{num_voices}_voices"
output_dir_full_path = os.path.join("/content/", output_folder_name)

# Create the output directory. If it already exists, clear its contents first.
if os.path.exists(output_dir_full_path):
    shutil.rmtree(output_dir_full_path) # Remove existing content for a clean run
os.makedirs(output_dir_full_path, exist_ok=True)
print(f"Output files will be saved to: {output_dir_full_path}")
# --- End Output Management Logic ---

# Check if the audio file exists
if not os.path.exists(audio_file):
    print(f"Error: Audio file '{audio_file}' not found. Please upload it to Colab.")
else:
    # Configure the separator to save output to the specific directory
    separator1.output_dir = output_dir_full_path

    # Perform the initial vocal/instrumental separation
    # separator1 and output_names are assumed to be loaded from 'Loading the models' cell
    # Files will now be saved in output_dir_full_path due to separator1.output_dir setting
    voc_instr_sep_files_full_paths = separator1.separate(audio_file, output_names)

    vocal_file_path = None
    for a_file_path in voc_instr_sep_files_full_paths:
        if 'vocals' in a_file_path.lower(): # Case-insensitive check for 'vocals'
            vocal_file_path = a_file_path
            break

    if vocal_file_path is None:
        print("Error: Vocal file not found after initial separation.")
    else:
        sep_voc_parts_to_transcribe = [] # This list will hold the paths to files ready for transcription

        # Conditional vocal separation based on num_voices
        if num_voices == 1:
            print("Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.")
            original_vocal_filename = os.path.basename(vocal_file_path)
            final_vocal_name_prefixed = f"{audio_base_name}_{num_voices}_{original_vocal_filename}"
            final_vocal_full_path = os.path.join(output_dir_full_path, final_vocal_name_prefixed)
            # Rename the vocal file within the output directory (both paths are absolute)
            os.rename(vocal_file_path, final_vocal_full_path)
            sep_voc_parts_to_transcribe.append(final_vocal_full_path)

        elif num_voices == 2:
            # model_libri2mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri2mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path]

        elif num_voices == 3:
            # model_libri3mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri3mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            final_part3_name = f"{audio_base_name}_{num_voices}_sep_voc_part3.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)
            part3_path = os.path.join(output_dir_full_path, final_part3_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            torchaudio.save(part3_path, est_sources[:, :, 2].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path, part3_path]
        else:
            print("Invalid number of voices specified. Please choose 1, 2 or 3.")

        # Conditional transcription
        if sep_voc_parts_to_transcribe:
            for i, part_file_to_transcribe in enumerate(sep_voc_parts_to_transcribe):
                # model_qwen3asr is assumed to be loaded from 'Loading the models' cell
                results = model_qwen3asr.transcribe(
                    audio=part_file_to_transcribe,
                    language=language, # set "English" to force the language
                )
                print()
                print(f"Lyrics of the singer #{i+1}: ")
                print(results[0].text)


Output files will be saved to: /content/mixed_combo_CEA_B_CEA_S_CEA_T_3_voices


100%|██████████| 2/2 [00:06<00:00,  3.31s/it]


Resampling the audio from 44100 Hz to 8000 Hz

Lyrics of the singer #1: 
Corte espadas afiladas, lenguas malas. Corte espadas afiladas, afiladas, lenguas malas, lenguas malas.

Lyrics of the singer #2: 
Fortunes fadas, afiladas, lenguas malas. Fortunes fadas, afiladas, lenguas malas, lenguas malas.

Lyrics of the singer #3: 
Corten espadas afiladas, lenguas malas. Corten espadas afiladas, lenguas malas, lenguas malas.


In [ ]:
# @title # Transcribe your song with 1, 2 or 3 voices!

import os
import shutil # Required for creating and clearing directories

# User inputs for audio file and language
audio_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

#@markdown ---
language = ""  # @param {type:"string"}
num_voices = 1 # @param ["1", "2", "3"] {type:"raw"}

# --- Output Management Logic ---
# Derive base name from audio_file (e.g., 'Pur_ti_miro_cut')
audio_base_name = os.path.splitext(os.path.basename(audio_file))[0]

# Define the output directory name based on audio and number of voices
output_folder_name = f"{audio_base_name}_{num_voices}_voices"
output_dir_full_path = os.path.join("/content/", output_folder_name)

# Create the output directory. If it already exists, clear its contents first.
if os.path.exists(output_dir_full_path):
    shutil.rmtree(output_dir_full_path) # Remove existing content for a clean run
os.makedirs(output_dir_full_path, exist_ok=True)
print(f"Output files will be saved to: {output_dir_full_path}")
# --- End Output Management Logic ---

# Check if the audio file exists
if not os.path.exists(audio_file):
    print(f"Error: Audio file '{audio_file}' not found. Please upload it to Colab.")
else:
    # Configure the separator to save output to the specific directory
    separator1.output_dir = output_dir_full_path

    # Perform the initial vocal/instrumental separation
    # separator1 and output_names are assumed to be loaded from 'Loading the models' cell
    # Files will now be saved in output_dir_full_path due to separator1.output_dir setting
    voc_instr_sep_files_full_paths = separator1.separate(audio_file, output_names)

    vocal_file_path = None
    for a_file_path in voc_instr_sep_files_full_paths:
        if 'vocals' in a_file_path.lower(): # Case-insensitive check for 'vocals'
            vocal_file_path = a_file_path
            break

    if vocal_file_path is None:
        print("Error: Vocal file not found after initial separation.")
    else:
        sep_voc_parts_to_transcribe = [] # This list will hold the paths to files ready for transcription

        # Conditional vocal separation based on num_voices
        if num_voices == 1:
            print("Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.")
            original_vocal_filename = os.path.basename(vocal_file_path)
            final_vocal_name_prefixed = f"{audio_base_name}_{num_voices}_{original_vocal_filename}"
            final_vocal_full_path = os.path.join(output_dir_full_path, final_vocal_name_prefixed)
            # Rename the vocal file within the output directory (both paths are absolute)
            os.rename(vocal_file_path, final_vocal_full_path)
            sep_voc_parts_to_transcribe.append(final_vocal_full_path)

        elif num_voices == 2:
            # model_libri2mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri2mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path]

        elif num_voices == 3:
            # model_libri3mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri3mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            final_part3_name = f"{audio_base_name}_{num_voices}_sep_voc_part3.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)
            part3_path = os.path.join(output_dir_full_path, final_part3_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            torchaudio.save(part3_path, est_sources[:, :, 2].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path, part3_path]
        else:
            print("Invalid number of voices specified. Please choose 1, 2 or 3.")

        # Conditional transcription
        if sep_voc_parts_to_transcribe:
            for i, part_file_to_transcribe in enumerate(sep_voc_parts_to_transcribe):
                # model_qwen3asr is assumed to be loaded from 'Loading the models' cell
                results = model_qwen3asr.transcribe(
                    audio=part_file_to_transcribe,
                    language=language, # set "English" to force the language
                )
                print()
                print(f"Lyrics of the singer #{i+1}: ")
                print(results[0].text)


Output files will be saved to: /content/mixed_combo_CEA_B_CEA_S_CEA_T_1_voices


100%|██████████| 2/2 [00:05<00:00,  2.80s/it]


Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.

Lyrics of the singer #1: 
Corte espadas, cortes peladas, afiladas, lenguas malas, lenguas malas.


In [ ]:
# @title # Transcribe your song with 1, 2 or 3 voices!

import os
import shutil # Required for creating and clearing directories

# User inputs for audio file and language
audio_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

#@markdown ---
language = "Spanish"  # @param {type:"string"}
num_voices = 3 # @param ["1", "2", "3"] {type:"raw"}

# --- Output Management Logic ---
# Derive base name from audio_file (e.g., 'Pur_ti_miro_cut')
audio_base_name = os.path.splitext(os.path.basename(audio_file))[0]

# Define the output directory name based on audio and number of voices
output_folder_name = f"{audio_base_name}_{num_voices}_voices"
output_dir_full_path = os.path.join("/content/", output_folder_name)

# Create the output directory. If it already exists, clear its contents first.
if os.path.exists(output_dir_full_path):
    shutil.rmtree(output_dir_full_path) # Remove existing content for a clean run
os.makedirs(output_dir_full_path, exist_ok=True)
print(f"Output files will be saved to: {output_dir_full_path}")
# --- End Output Management Logic ---

# Check if the audio file exists
if not os.path.exists(audio_file):
    print(f"Error: Audio file '{audio_file}' not found. Please upload it to Colab.")
else:
    # Configure the separator to save output to the specific directory
    separator1.output_dir = output_dir_full_path

    # Perform the initial vocal/instrumental separation
    # separator1 and output_names are assumed to be loaded from 'Loading the models' cell
    # Files will now be saved in output_dir_full_path due to separator1.output_dir setting
    voc_instr_sep_files_full_paths = separator1.separate(audio_file, output_names)

    vocal_file_path = None
    for a_file_path in voc_instr_sep_files_full_paths:
        if 'vocals' in a_file_path.lower(): # Case-insensitive check for 'vocals'
            vocal_file_path = a_file_path
            break

    if vocal_file_path is None:
        print("Error: Vocal file not found after initial separation.")
    else:
        sep_voc_parts_to_transcribe = [] # This list will hold the paths to files ready for transcription

        # Conditional vocal separation based on num_voices
        if num_voices == 1:
            print("Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.")
            original_vocal_filename = os.path.basename(vocal_file_path)
            final_vocal_name_prefixed = f"{audio_base_name}_{num_voices}_{original_vocal_filename}"
            final_vocal_full_path = os.path.join(output_dir_full_path, final_vocal_name_prefixed)
            # Rename the vocal file within the output directory (both paths are absolute)
            os.rename(vocal_file_path, final_vocal_full_path)
            sep_voc_parts_to_transcribe.append(final_vocal_full_path)

        elif num_voices == 2:
            # model_libri2mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri2mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path]

        elif num_voices == 3:
            # model_libri3mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri3mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            final_part3_name = f"{audio_base_name}_{num_voices}_sep_voc_part3.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)
            part3_path = os.path.join(output_dir_full_path, final_part3_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            torchaudio.save(part3_path, est_sources[:, :, 2].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path, part3_path]
        else:
            print("Invalid number of voices specified. Please choose 1, 2 or 3.")

        # Conditional transcription
        if sep_voc_parts_to_transcribe:
            for i, part_file_to_transcribe in enumerate(sep_voc_parts_to_transcribe):
                # model_qwen3asr is assumed to be loaded from 'Loading the models' cell
                results = model_qwen3asr.transcribe(
                    audio=part_file_to_transcribe,
                    language=language, # set "English" to force the language
                )
                print()
                print(f"Lyrics of the singer #{i+1}: ")
                print(results[0].text)


Output files will be saved to: /content/mixed_combo_CEA_B_CEA_S_CEA_T_3_voices


100%|██████████| 2/2 [00:05<00:00,  2.86s/it]


Resampling the audio from 44100 Hz to 8000 Hz

Lyrics of the singer #1: 
Corte en espadas afiladas, lenguas malas. Corte en espadas afiladas, afiladas, lenguas malas, lenguas malas.

Lyrics of the singer #2: 
Fortunes faldas afiladas, lenguas malas. Fortunes faldas afiladas, lenguas malas, lenguas malas.

Lyrics of the singer #3: 
Corten espadas afiladas, lenguas malas. Corten espadas afiladas, lenguas malas, lenguas malas.


In [ ]:
# @title # Transcribe your song with 1, 2 or 3 voices!

import os
import shutil # Required for creating and clearing directories

# User inputs for audio file and language
audio_file = "/content/mixed_combo_CEA_B_CEA_S_CEA_T.wav"  # @param {type:"string"}

#@markdown ---
language = "Spanish"  # @param {type:"string"}
num_voices = 1 # @param ["1", "2", "3"] {type:"raw"}

# --- Output Management Logic ---
# Derive base name from audio_file (e.g., 'Pur_ti_miro_cut')
audio_base_name = os.path.splitext(os.path.basename(audio_file))[0]

# Define the output directory name based on audio and number of voices
output_folder_name = f"{audio_base_name}_{num_voices}_voices"
output_dir_full_path = os.path.join("/content/", output_folder_name)

# Create the output directory. If it already exists, clear its contents first.
if os.path.exists(output_dir_full_path):
    shutil.rmtree(output_dir_full_path) # Remove existing content for a clean run
os.makedirs(output_dir_full_path, exist_ok=True)
print(f"Output files will be saved to: {output_dir_full_path}")
# --- End Output Management Logic ---

# Check if the audio file exists
if not os.path.exists(audio_file):
    print(f"Error: Audio file '{audio_file}' not found. Please upload it to Colab.")
else:
    # Configure the separator to save output to the specific directory
    separator1.output_dir = output_dir_full_path

    # Perform the initial vocal/instrumental separation
    # separator1 and output_names are assumed to be loaded from 'Loading the models' cell
    # Files will now be saved in output_dir_full_path due to separator1.output_dir setting
    voc_instr_sep_files_full_paths = separator1.separate(audio_file, output_names)

    vocal_file_path = None
    for a_file_path in voc_instr_sep_files_full_paths:
        if 'vocals' in a_file_path.lower(): # Case-insensitive check for 'vocals'
            vocal_file_path = a_file_path
            break

    if vocal_file_path is None:
        print("Error: Vocal file not found after initial separation.")
    else:
        sep_voc_parts_to_transcribe = [] # This list will hold the paths to files ready for transcription

        # Conditional vocal separation based on num_voices
        if num_voices == 1:
            print("Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.")
            original_vocal_filename = os.path.basename(vocal_file_path)
            final_vocal_name_prefixed = f"{audio_base_name}_{num_voices}_{original_vocal_filename}"
            final_vocal_full_path = os.path.join(output_dir_full_path, final_vocal_name_prefixed)
            # Rename the vocal file within the output directory (both paths are absolute)
            os.rename(vocal_file_path, final_vocal_full_path)
            sep_voc_parts_to_transcribe.append(final_vocal_full_path)

        elif num_voices == 2:
            # model_libri2mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri2mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path]

        elif num_voices == 3:
            # model_libri3mix is assumed to be loaded from 'Loading the models' cell
            est_sources = model_libri3mix.separate_file(path=vocal_file_path)

            # Define prefixed names and full paths for saved parts
            final_part1_name = f"{audio_base_name}_{num_voices}_sep_voc_part1.wav"
            final_part2_name = f"{audio_base_name}_{num_voices}_sep_voc_part2.wav"
            final_part3_name = f"{audio_base_name}_{num_voices}_sep_voc_part3.wav"
            part1_path = os.path.join(output_dir_full_path, final_part1_name)
            part2_path = os.path.join(output_dir_full_path, final_part2_name)
            part3_path = os.path.join(output_dir_full_path, final_part3_name)

            # Save the separated parts to the output directory with new names
            torchaudio.save(part1_path, est_sources[:, :, 0].detach().cpu(), 8000)
            torchaudio.save(part2_path, est_sources[:, :, 1].detach().cpu(), 8000)
            torchaudio.save(part3_path, est_sources[:, :, 2].detach().cpu(), 8000)
            sep_voc_parts_to_transcribe = [part1_path, part2_path, part3_path]
        else:
            print("Invalid number of voices specified. Please choose 1, 2 or 3.")

        # Conditional transcription
        if sep_voc_parts_to_transcribe:
            for i, part_file_to_transcribe in enumerate(sep_voc_parts_to_transcribe):
                # model_qwen3asr is assumed to be loaded from 'Loading the models' cell
                results = model_qwen3asr.transcribe(
                    audio=part_file_to_transcribe,
                    language=language, # set "English" to force the language
                )
                print()
                print(f"Lyrics of the singer #{i+1}: ")
                print(results[0].text)


Output files will be saved to: /content/mixed_combo_CEA_B_CEA_S_CEA_T_1_voices


100%|██████████| 2/2 [00:05<00:00,  2.88s/it]


Skipping singing source separation for 1 voice. Transcribing directly from main vocal track.

Lyrics of the singer #1: 
Corte espadas, corte espadas afiladas, lenguas malas, corte espadas, corte espadas afiladas, lenguas malas, lenguas malas.
